# DepthWizard v4 — GAMUS (PNG mirror), Kaggle 2×T4

**Accelerator `GPU T4 x2` · Internet ON · Add-ons → Secrets: `HF_TOKEN` *and*
`GITHUB_PAT`.**

`DepthWizard` is a private repo, so the clone needs a GitHub personal access
token — a fine-grained token with *Contents: read* on this one repo, or a
classic token with `repo`. The notebook clones into `/kaggle/temp`, not
`/kaggle/working`: `git clone` stores the token-bearing URL in `.git/config`,
and `/kaggle/working` is published as the notebook's output dataset. It also
resets `origin` to the clean URL, and scrubs the token out of any clone error
before printing it.
Start the real run with **Save Version → Save & Run All (Commit)** — the headless
commit gets the full ~12 h and persists `/kaggle/working`; an interactive session
idles out long before the run ends.

Attach the GAMUS PNG dataset (`{train,val,test}/{images,depth}/gamus_*.png`) as
the only input. Nothing is downloaded: `prepare_data.py` pulls one HDF5 per tile
from the gated `earthflow/GAMUS` repo, but the attached mirror is already
unpacked, so cell 1 packs those PNGs straight into the memmap shard store that
`train.py` reads.

**Set `SMOKE = True` for the first run** (~5 min). Success is a clean exit, *one*
set of loss lines in the log — which is what proves the rank-1 stdout redirect
worked — and `best.pt` / `metrics.json` written once.

### The one thing to check in the output
The `HEIGHT CHECK` line. A PNG mirror's height encoding cannot be assumed, so
the packer guesses from the dtype and prints what it produced. GAMUS AGL should
read median ~2–5 m, p99 ~30–60 m, max ≲100 m. If everything is 100× too large it
is centimetres — set `DEPTH_SCALE = "0.01"` and re-run.

### Two things that differ from the H100 profile
* **No class masks** ship with the PNG mirror, so every pixel packs as
  `NO_LABEL`. `seg_ce_loss` short-circuits an all-unlabelled batch to zero, so
  the 0.2-weight semantic head contributes nothing; per-class metrics in
  `metrics.json` come out empty. Depth training is unaffected.
* **SynRS3D is dropped** (`--datasets gamus`), so every source tile is 0.33 m
  GSD. The 0.30–1.20 m jitter still applies, capped per tile by
  `achievable_gsd_range`.


In [ ]:
# ═══ DepthWizard v4 — GAMUS PNG mirror on Kaggle 2×T4 — run this one cell ═══
import os, sys, glob, subprocess, pathlib

REPO   = "/kaggle/temp/DepthWizard"   # NOT /kaggle/working: a clone of a private repo
                                      # carries the PAT in .git/config, and /kaggle/working
                                      # is published as the notebook's output dataset
CODE   = f"{REPO}/FineTunning/V4_Kaggle"
GAMUS  = "/kaggle/input"          # autodetects the directory holding train/images
DATA   = "/kaggle/temp/dwdata"    # packed store — off the 20 GB output quota
OUT    = "/kaggle/working/outputs/v4"
SMOKE  = False                    # True first: ~5 min, proves DDP + data + model end-to-end
RESUME = ""                       # session 2: "/kaggle/input/<prev>/outputs/v4/last_full.pt"
DEPTH_SCALE = "auto"              # "auto" | "0.01" (cm→m) | "1.0" (already metres)
TRAIN_TILES = 0                   # 0 = all; lower it if /kaggle/temp fills
MAX_MINUTES = 960                 # total training budget across ALL sessions
SYNRS3D_ARCHIVES = 4              # 0 to skip.  1-3 are the whole g1 family
                                  # (0.6-1.0 m, the only GSD range GAMUS's
                                  # 1024px/0.33m tiles cannot reach); 4 adds the
                                  # g05 archive v4-2 trained on.  They pack into
                                  # SEPARATE stores, one per GSD family.
DFC23_DIR = ""                    # e.g. "/kaggle/input/dfc23-track2-height-estimation/track2/train"
                                  # real 0.5 m satellite optical + real nDSM.
                                  # "" to skip.
RUN_TESTS = True                  # pytest before training: a broken commit then costs
                                  # minutes instead of the 8 h it cost twice already
EPOCHS = 24                       # the MEASURED T4 rate, not an aspiration — see below
SESSION_MINUTES = 480             # cap on THIS session; 960/480 = two sessions

GIT_URL = "github.com/AbhayKale332/DepthWizard.git"
sh = lambda c: subprocess.run(c, shell=True, check=True)

def clone():
    """The repo is private, so this needs GITHUB_PAT (Add-ons -> Secrets, `repo` scope).

    List form and captured output, never an f-string through the shell: a failing
    clone prints the URL it tried, and that URL contains the token.
    """
    pat = ""
    try:
        from kaggle_secrets import UserSecretsClient
        pat = UserSecretsClient().get_secret("GITHUB_PAT")
    except Exception as e:
        print("no GITHUB_PAT secret (", e, ") - trying an anonymous clone")
    url = f"https://x-access-token:{pat}@{GIT_URL}" if pat else f"https://{GIT_URL}"
    r = subprocess.run(["git", "clone", "--depth", "1", url, REPO],
                       capture_output=True, text=True)
    if r.returncode:
        raise SystemExit("git clone failed: "
                         + (r.stderr.replace(pat, "***") if pat else r.stderr))
    # `git clone` writes the token-bearing URL into .git/config. Drop it.
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin",
                    f"https://{GIT_URL}"], check=True)
    print("cloned", REPO)

if not os.path.isdir(REPO):
    clone()
os.chdir(CODE); sys.path.insert(0, CODE)
sh("pip install -q -r requirements.txt")

# ── 0b. Tests ─────────────────────────────────────────────────────────────
# pytest is in requirements.txt and nothing here downloads weights
# (tests/test_real_dinov3.py builds a 6-layer DINOv3 from random init), so this
# needs no HF token and no network.  `sh` uses check=True, so a failure raises
# and the cell stops before anything is packed or trained.
if RUN_TESTS:
    # Not `sh`: check=True raises a CalledProcessError whose traceback is what
    # Jupyter shows, and pytest's own collection errors scroll away above it.
    # Capture, print the tail, then fail with the reason visible.
    r = subprocess.run("python -m pytest -q --tb=short -rf", shell=True,
                       capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    print(out[-8000:])
    if r.returncode:
        raise SystemExit(
            f"pytest exited {r.returncode} "
            f"({'collection/usage error' if r.returncode == 2 else 'test failures'}). "
            f"Output above. Set RUN_TESTS = False to skip.")

try:                               # the DINOv3-SAT encoder is gated too
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set")
except Exception as e:
    print("!! no HF_TOKEN secret:", e)

# ── 1. PNG tree → memmap shard store ───────────────────────────────────────
# train.py only ever reads the packed store; prepare_data.py builds it from the
# gated HDF5 repo, which is not what is attached here. Same store, local PNGs.
PACKER = pathlib.Path(CODE, "pack_gamus_png.py")
if not PACKER.exists():
    raise SystemExit(f"{PACKER} missing - the clone is stale; delete {REPO} and re-run")
# What the attached mirror actually looks like — directory names vary per mirror,
# and a wrong guess surfaces as "no image/depth pairs".
for d in sorted(pathlib.Path(GAMUS).glob("**/train")):
    print(d)
    for sub in sorted(x for x in d.iterdir() if x.is_dir()):
        names = [f.name for f in sorted(sub.iterdir())[:3]]
        print("   ", sub.name + "/", len(list(sub.iterdir())), "entries", names)

sh(f"python pack_gamus_png.py --src {GAMUS} --out {DATA} "
   f"--depth_scale {DEPTH_SCALE} --train_tiles {TRAIN_TILES}")

# ── 1b. SynRS3D ───────────────────────────────────────────────────────────
# v3 trained on gamus + synrs3d (12,266 tiles) and scored 2.723 m; the first
# two v4 runs trained on gamus alone (3,453 tiles) and scored 3.959 / 3.441.
# SynRS3D is 8,429 synthetic tiles already at 512 px / 0.5 m, and it is also
# the only cross-sensor defence in the mix — which is what ISRO's own imagery
# will actually test.  One archive is ~4 GB and downloads in a few minutes.
if SYNRS3D_ARCHIVES:
    sh(f"python prepare_data.py --data_root {DATA} --datasets synrs3d "
       f"--synrs3d_archives {SYNRS3D_ARCHIVES}")

# ── 1c. DFC23 Track 2 ─────────────────────────────────────────────────────
# The only source in the mix that is the actual deployment domain: real 0.5 m
# satellite optical with a real nDSM.  GAMUS is US aerial, SynRS3D is synthetic.
# SAR is on disk and deliberately unused — the store is 3-channel and a
# Cartosat-class deployment has no SAR at inference.
#
# Only track2/train carries a reference nDSM (the contest withheld val/test
# labels), so the val split is carved out of those scenes BY SCENE, not by tile.
if DFC23_DIR:
    # Look before packing: the Kaggle file listing is truncated, so scene count,
    # raster size and the nDSM value range are all unknown until measured. The
    # packer prints its own HEIGHT CHECK after, but a bad --dfc23_dir should
    # fail here rather than silently pack nothing.
    for sub in ("rgb", "dsm", "sar"):
        fs = sorted(glob.glob(f"{DFC23_DIR}/{sub}/*.tif"))
        print(f"  {sub}/: {len(fs)} tif", fs[:2])
    try:
        import rasterio
        f0 = sorted(glob.glob(f"{DFC23_DIR}/rgb/*.tif"))[0]
        d0 = sorted(glob.glob(f"{DFC23_DIR}/dsm/*.tif"))[0]
        for tag, f in (("rgb", f0), ("dsm", d0)):
            with rasterio.open(f) as ds:
                print(f"  {tag}: {ds.width}x{ds.height} {ds.count}band "
                      f"{ds.dtypes[0]} crs={ds.crs} px={ds.transform.a:.3f}")
        with rasterio.open(d0) as ds:
            import numpy as _np
            h = ds.read(1).astype("float32")
            h = h[_np.isfinite(h)]
            print(f"  dsm values: min {h.min():.1f}  median {_np.median(h):.2f}  "
                  f"p99 {_np.percentile(h, 99):.1f}  max {h.max():.1f}  "
                  f"(nDSM -> median near 0)")
    except Exception as e:
        print("  probe failed:", e)
    sh(f"python prepare_data.py --data_root {DATA} --datasets dfc23 "
       f"--dfc23_dir {DFC23_DIR}")

sh(f"du -sh {DATA}; df -h /kaggle/temp | tail -1")

# ── 2. Train: torchrun, one process per T4 ──────────────────────────────
os.environ.update(PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
                  NCCL_P2P_DISABLE="1",          # no NVLink on Kaggle's T4 pair
                  OMP_NUM_THREADS="1", TOKENIZERS_PARALLELISM="false")
# gamus FIRST and it must stay first: loaders.py takes the first source with a
# val split as the primary val set, that is the 400-tile prefix every run since
# v1 is scored on, and best.pt is selected on it.  Every other prepared val
# store is scored and reported alongside it, never selected on.
_srcs = ["gamus"]
if SYNRS3D_ARCHIVES:
    _srcs += ["synrs3d_g1"] if SYNRS3D_ARCHIVES <= 3 else ["synrs3d_g1", "synrs3d_g05"]
if DFC23_DIR:
    _srcs += [pathlib.Path(d).parent.name for d in
              sorted(glob.glob(f"{DATA}/dfc23_*/train/index.json"))]
DATASETS = ",".join(_srcs)
print("datasets:", DATASETS)
args = (f"--data_root {DATA} --output_dir {OUT} --datasets {DATASETS} "
        # Turing has no bf16 tensor cores -> fp16 + GradScaler; ViT-L needs
        # encoder checkpointing to fit 15 GB (static VRAM is ~8.3 GB before activations).
        "--amp_dtype fp16 --grad_checkpoint_encoder true "
        "--batch_size 2 --grad_accum 6 --eval_batch_mult 8 "      # 2 ranks x 2 x 6 = batch 24
        "--num_workers 2 --prefetch_factor 2 --compile_model false "
        # Freeze the bottom 8 blocks.  The previous run unfroze all 24 at
        # llrd 0.90, so the bottom block ran at 6e-5 * 0.9**24 = 4.8e-6 — it
        # paid full price in gradients, all-reduce and AdamW state to move
        # essentially not at all, and throughput halved (4.9 -> 2.3 img/s).
        "--encoder_unfreeze_blocks 16 --llrd 0.90 "
        # MAX_MINUTES is the budget the cosine is SIZED for, not a safety cap:
        # progress is max(epoch_frac, elapsed/max_minutes).  960/480 is a
        # two-session run and EVERY flag here must be identical in both; only
        # RESUME differs.  24 epochs is the measured rate (~22 min frozen,
        # ~40 min after the unfreeze), so the epoch term and the clock term land
        # together instead of one being dead all run.
        #
        # v4-2 went out as 600/16 with session_minutes 480 and save_full_state
        # false: it stopped at 81 % of the cosine with lr still 5.09e-05, could
        # not be resumed, and scored 3.804 m against v4's 3.441 m.  Read the
        # [sched] line train.py now prints at startup — it says up front where
        # the anneal will stop.
        f"--epochs {EPOCHS} --max_minutes {MAX_MINUTES} "
        f"--session_minutes {SESSION_MINUTES} "
        # --eval_every 1: 6 selection points is too coarse to pick best.pt when
        # d1 is non-monotonic across evals while RMSE falls.  An eval is ~1.5 min
        # against ~43 min of training.
        "--eval_every 1 "
        # w_seg is left at config.py's 0.2 and NOT zeroed here.  The `--w_seg 0`
        # this used to carry was reasoned from the GAMUS PNG mirror shipping no
        # class rasters, which was true when GAMUS was the only source.  SynRS3D
        # does ship them ([data] prints `seg=yes`), so v4-2 computed a real seg
        # CE on every mixed batch and multiplied it by zero — the `seg=2.13` in
        # the log is the raw unweighted value, never in the gradient.  Restoring
        # it also gives flatness_loss back its ground/road/water restriction.
        # loaders.py warns at startup if NO store has labels.
        # ~6 GB/epoch, ~30 s to write (under 3 % overhead) — and without it a
        # session that dies mid-anneal is unrecoverable, which is exactly what
        # cost v4-2 its last three epochs.  Required for the two-session profile.
        "--save_full_state true --make_zip false")                # the zip blows the 20 GB quota
if SMOKE:
    args = f"--smoke --data_root {DATA} --output_dir {OUT} --datasets gamus --make_zip false"
if RESUME:
    args += f" --resume {RESUME}"
sh(f"torchrun --standalone --nnodes=1 --nproc_per_node=2 train.py {args}")


## If the session dies

`last_full.pt` carries the optimiser, scaler, EMA, teacher, RNG and elapsed
minutes. Attach the dead run's version as an input to a fresh copy of this
notebook and set `RESUME = "/kaggle/input/<name>/outputs/v4/last_full.pt"`.
`run.log` should show the LR continuing *down* the cosine, not re-warming.
Keep `MAX_MINUTES` the same in both sessions — it is the total budget that
drives the cosine; `session_minutes` is what caps one session.

`/kaggle/temp` is wiped between sessions, so a resumed run re-clones the repo
and re-packs the PNGs first (a few minutes). That is deliberate: the store does not belong in
`/kaggle/working`, where it would eat the 20 GB output quota.

If training finished but the exports did not, run `bash run_kaggle.sh finalize`
in a short single-GPU notebook against the committed `best.pt`.
